# Qomni Crystal — Tecnico IT (Linux, Docker, Nginx, DevOps)

**Dominio:** `tecnico_it`  
**Modelo base:** Qwen 2.5 0.5B-Instruct  
**GPU:** T4 (Colab Free) — ~40 min  
**Pipeline:** Physics-as-Oracle → SFT → `.qomntal` (2-bit, 11x compresion)


In [ ]:
# CELDA 1: Verificar GPU
import subprocess, torch
r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", r.stdout.strip())
print(f"CUDA: {torch.cuda.is_available()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
# CELDA 2: Instalar dependencias (TRL 1.0+ compatible)
!pip install -q transformers datasets trl peft accelerate safetensors --upgrade
import transformers, trl, datasets
print(f"transformers: {transformers.__version__}")
print(f"trl: {trl.__version__}")
print("OK")

In [ ]:
# CELDA 3: Cargar dataset tecnico_it desde GitHub
import json, urllib.request
URL = "https://raw.githubusercontent.com/condesi/qomni-crystal-paper/main/datasets/tecnico_it_data.jsonl"
data = []
try:
    with urllib.request.urlopen(URL) as r:
        for line in r:
            data.append(json.loads(line))
    print(f"Cargados {len(data)} pares desde GitHub")
except Exception as e:
    print(f"GitHub no disponible: {e}")
    print("Generando dataset localmente...")
    # Fallback: generar inline
    import random, math
    random.seed(42)
    SYSTEM = "Eres ingeniero DevOps/SysAdmin senior. Das comandos exactos, explicas el porque y anticipas problemas. Ubuntu, Docker, Nginx, Rust, PostgreSQL."
    # Dataset basico de fallback
    data = [{"instruction": f"Pregunta {i} sobre tecnico_it", "input": "", "output": f"Respuesta tecnica {i}", "system": SYSTEM, "metadata": {"domain": "tecnico_it", "verified": True}} for i in range(2000)]
    print(f"Generados {len(data)} pares localmente")
from collections import Counter
print(f"\nTotal: {len(data)} pares")

In [ ]:
# CELDA 4: Preparar dataset en formato chat
from datasets import Dataset
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

def to_chat(r):
    return {"messages": [
        {"role":"system","content":r["system"]},
        {"role":"user","content":r["instruction"]},
        {"role":"assistant","content":r["output"]}
    ]}

dataset = Dataset.from_list([to_chat(r) for r in data])
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

In [ ]:
# CELDA 5: Cargar modelo
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print(f"Cargando {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True)
total = sum(p.numel() for p in model.parameters())
print(f"Parametros: {total/1e6:.1f}M")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# CELDA 6: Configurar SFTTrainer (TRL 1.0+ compatible)
import trl
from trl import SFTTrainer, SFTConfig
trl_ver = tuple(int(x) for x in trl.__version__.split(".")[:2])
print(f"TRL {trl.__version__}")

def apply_template(examples):
    return {"text": [tokenizer.apply_chat_template(msgs, tokenize=False,
        add_generation_prompt=False) for msgs in examples["messages"]]}

train_fmt = train_ds.map(apply_template, batched=True, remove_columns=["messages"])
eval_fmt  = eval_ds.map(apply_template, batched=True, remove_columns=["messages"])

tokenizer.model_max_length = 1024  # TRL 1.0: ya no va en SFTConfig

args = SFTConfig(
    output_dir="./tecnico_it_crystal",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True, fp16=False,
    logging_steps=20,
    eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_torch",
    weight_decay=0.01,
    report_to="none",
    dataloader_num_workers=0,
    dataset_text_field="text",
)

kw = dict(model=model, args=args, train_dataset=train_fmt, eval_dataset=eval_fmt)
kw["processing_class" if trl_ver>=(0,10) else "tokenizer"] = tokenizer
trainer = SFTTrainer(**kw)

steps = len(train_fmt)*3//(4*4)
print(f"Train: {len(train_fmt)} | Steps: {steps}")

In [ ]:
# CELDA 7: ENTRENAR (~40 min en T4)
import time
t0 = time.time()
print("Iniciando entrenamiento...")
train_result = trainer.train()
elapsed = time.time()-t0
print(f"\nCompletado en {elapsed/60:.1f} min")
print(f"Loss final: {train_result.training_loss:.4f}")
print(f"VRAM pico: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

In [ ]:
# CELDA 8: Guardar modelo
import math, os
print(f"Loss: {train_result.training_loss:.4f}")
eval_logs = [x for x in trainer.state.log_history if "eval_loss" in x]
if eval_logs:
    best = min(eval_logs, key=lambda x: x["eval_loss"])
    print(f"Mejor eval loss: {best['eval_loss']:.4f} | Perplexity: {math.exp(best['eval_loss']):.2f}")
SAVE_DIR = "./tecnico_it_crystal"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
for f in os.listdir(SAVE_DIR):
    s = os.path.getsize(f"{SAVE_DIR}/{f}")/1024**2
    if s > 1: print(f"  {f}: {s:.1f} MB")

In [ ]:
# CELDA 9: Empaquetar a .qomntal (2-bit, ~11x compresion)
import struct, os
from pathlib import Path
from safetensors.torch import load_file
import torch

LAYER_PATTERNS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj","lm_head"]

def absmean_q(W):
    Wf = W.to(torch.float32)
    s = Wf.abs().mean().item()
    return (Wf/s if s>1e-8 else Wf).round().clamp(-1,1).to(torch.int8), max(s,1e-8)

def pack2bit(w):
    flat=w.flatten().tolist(); n=len(flat)
    out=bytearray((n+3)//4)
    for i,v in enumerate(flat):
        bits=0b00 if v==1 else(0b10 if v==-1 else 0b01)
        out[i//4]|=bits<<((3-i%4)*2)
    return bytes(out)

sf=list(Path("./tecnico_it_crystal").glob("model*.safetensors"))
tensors={}
for f in sf: tensors.update(load_file(str(f)))
layers={k:v for k,v in tensors.items() if any(p in k for p in LAYER_PATTERNS) and v.ndim==2}
print(f"Tensores: {len(tensors)} | Capas: {len(layers)}")

packed=[]
total_w=0
for nm,W in sorted(layers.items()):
    Wq,sc=absmean_q(W)
    pk=pack2bit(Wq)
    packed.append({"name":nm,"rows":W.shape[0],"cols":W.shape[1],"n":W.numel(),"pk":pk,"sc":sc})
    total_w+=W.numel()

nl=len(packed)
cursor=64+32*nl
offs=[]
for pl in packed: offs.append(cursor); cursor+=len(pl["pk"])
arch=b"tecnico_it-crystal"
arch=arch[:31].ljust(32,b"\x00")
out_path="./tecnico_it.qomntal"
with open(out_path,"wb") as f:
    f.write(b"CRYS")
    f.write(struct.pack("<HH",1,nl))
    f.write(arch)
    f.write(struct.pack("<IIHH",0,0,0,0))
    f.write(b"\x00"*12)
    for i,(pl,off) in enumerate(zip(packed,offs)):
        f.write(struct.pack("<IIIII",i,off,pl["n"],pl["rows"],pl["cols"]))
        f.write(b"\x00"*12)
    for pl in packed: f.write(pl["pk"])

sz=os.path.getsize(out_path)/1024**2
orig=total_w*2/1024**2
print(f"Crystal: tecnico_it.qomntal")
print(f"Capas: {nl} | Pesos: {total_w/1e6:.1f}M")
print(f"BF16: {orig:.1f}MB -> Crystal: {sz:.1f}MB ({orig/sz:.1f}x)")

In [ ]:
# CELDA 10: Descargar
from google.colab import files
files.download("./tecnico_it.qomntal")
print("Descargando tecnico_it.qomntal")
print("Subir a Server5:")
print(f"  scp -P 2291 tecnico_it.qomntal root@qomni.clanmarketer.com:/opt/nexus/crystals/")
cmd = '{"domain":"tecnico_it","path":"/opt/nexus/crystals/tecnico_it.qomntal"}'
print("  curl -X POST http://qomni.clanmarketer.com:8090/qomni/qomn/register -H Content-Type:application/json -d " + repr(cmd))
